# 02. Data Visualization

Quick visual checks for prices and value factors from `data/research.db`.

In [ ]:
from __future__ import annotations

import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "data" / "research.db").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DB_PATH = PROJECT_ROOT / "data" / "research.db"
assert DB_PATH.exists(), f"research.db not found: {DB_PATH}"
DB_PATH

In [ ]:
def read_sql(sql: str, params: tuple = ()) -> pd.DataFrame:
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)


def price_history(code: str, start: str = "2016-01-01") -> pd.DataFrame:
    df = read_sql(
        """
        SELECT date, close, COALESCE(adj_close, close) AS adj_close, volume
        FROM prices_daily
        WHERE stock_code = ? AND date >= ?
        ORDER BY date
        """,
        (code.zfill(6), start),
    )
    df["date"] = pd.to_datetime(df["date"])
    return df


def latest_factor_snapshot(as_of: str) -> pd.DataFrame:
    df = read_sql(
        """
        WITH latest_prices AS (
          SELECT stock_code, date AS price_date, COALESCE(adj_close, close) AS price, volume,
                 ROW_NUMBER() OVER (PARTITION BY stock_code ORDER BY date DESC) AS rn
          FROM prices_daily
          WHERE date <= ?
        ), latest_financials AS (
          SELECT stock_code, fiscal_period, disclosed_at, eps, bps, net_income, total_equity,
                 ROW_NUMBER() OVER (PARTITION BY stock_code ORDER BY disclosed_at DESC, fiscal_period DESC, id DESC) AS rn
          FROM financials
          WHERE disclosed_at <= ?
        )
        SELECT s.code, s.name, s.market, lp.price_date, lp.price, lp.volume,
               lf.fiscal_period, lf.disclosed_at, lf.eps, lf.bps, lf.net_income, lf.total_equity
        FROM stocks s
        LEFT JOIN latest_prices lp ON lp.stock_code = s.code AND lp.rn = 1
        LEFT JOIN latest_financials lf ON lf.stock_code = s.code AND lf.rn = 1
        WHERE s.market = 'KOSPI'
          AND s.listed_at <= ?
          AND (s.delisted_at IS NULL OR s.delisted_at > ?)
        """,
        (as_of, as_of, as_of, as_of),
    )
    df["per"] = pd.to_numeric(df["price"], errors="coerce") / pd.to_numeric(df["eps"], errors="coerce")
    df["pbr"] = pd.to_numeric(df["price"], errors="coerce") / pd.to_numeric(df["bps"], errors="coerce")
    df["roe"] = pd.to_numeric(df["net_income"], errors="coerce") / pd.to_numeric(df["total_equity"], errors="coerce")
    df.loc[df["eps"] <= 0, "per"] = np.nan
    df.loc[df["bps"] <= 0, "pbr"] = np.nan
    df.loc[df["total_equity"] <= 0, "roe"] = np.nan
    df = df.replace([np.inf, -np.inf], np.nan)
    return df

## 10-year price trend

In [ ]:
code = "005930"  # Samsung Electronics
px = price_history(code)

fig, ax = plt.subplots()
sns.lineplot(data=px, x="date", y="adj_close", ax=ax)
ax.set_title(f"{code} adjusted close")
ax.set_xlabel("Date")
ax.set_ylabel("Price")
plt.show()

## PER / PBR / ROE distributions

In [ ]:
as_of = "2025-12-31"
snapshot = latest_factor_snapshot(as_of)
clean = snapshot.dropna(subset=["per", "pbr", "roe"]).copy()
clean = clean[(clean["per"].between(0, 80)) & (clean["pbr"].between(0, 10)) & (clean["roe"].between(-1, 1))]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(clean["per"], bins=40, kde=True, ax=axes[0])
sns.histplot(clean["pbr"], bins=40, kde=True, ax=axes[1])
sns.histplot(clean["roe"], bins=40, kde=True, ax=axes[2])
axes[0].set_title("PER distribution")
axes[1].set_title("PBR distribution")
axes[2].set_title("ROE distribution")
plt.tight_layout()
plt.show()

## Value factor scatter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sns.scatterplot(data=clean, x="per", y="pbr", hue="roe", palette="viridis", size="volume", sizes=(20, 250), alpha=0.75, ax=ax)
ax.set_title(f"PER vs PBR colored by ROE ({as_of})")
ax.set_xlabel("PER")
ax.set_ylabel("PBR")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Top candidates by simple value score

In [ ]:
def zscore(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    std = values.std(ddof=0)
    return (values - values.mean()) / std if std and not pd.isna(std) else values * 0


ranked = clean.copy()
ranked["score"] = -zscore(ranked["per"]) - 0.8 * zscore(ranked["pbr"]) + zscore(ranked["roe"])
ranked.sort_values("score", ascending=False)[["code", "name", "per", "pbr", "roe", "score"]].head(20)